# EDA with SQL

The launch table is loaded into an in-memory SQLite database (`SPACEXTBL`) and the ten capstone queries are run with pandas.

In [1]:
import sqlite3, pandas as pd
df=pd.read_csv("data/spacex_launch_table.csv"); df.columns=[c.strip() for c in df.columns]
for c in df.select_dtypes("object"): df[c]=df[c].str.strip()
con=sqlite3.connect(":memory:"); df.to_sql("SPACEXTBL",con,index=False)
q=lambda sql: pd.read_sql(sql,con)

/tmp/ipykernel_128/171966683.py:3: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for c in df.select_dtypes("object"): df[c]=df[c].str.strip()


### 1. Unique launch sites

In [2]:
q("SELECT DISTINCT Launch_Site FROM SPACEXTBL")

,Launch_Site
0,CCAFS LC-40
1,VAFB SLC-4E
2,KSC LC-39A
3,CCAFS SLC-40


### 2. Five records where launch site begins with `CCA`

In [3]:
q("SELECT * FROM SPACEXTBL WHERE Launch_Site LIKE 'CCA%' LIMIT 5")

,Date,Time (UTC),Booster_Version,Launch_Site,Payload,PAYLOAD_MASS__KG_,Orbit,Customer,Mission_Outcome,Landing_Outcome
0,2010-06-04,18:45:00,F9 v1.0 B0003,CCAFS LC-40,Dragon Spacecraft Qualification Unit,0,LEO,SpaceX,Success,Failure (parachute)
1,2010-12-08,15:43:00,F9 v1.0 B0004,CCAFS LC-40,"Dragon demo flight C1, two CubeSats, barrel of...",0,LEO (ISS),NASA (COTS) NRO,Success,Failure (parachute)
2,2012-05-22,7:44:00,F9 v1.0 B0005,CCAFS LC-40,Dragon demo flight C2,525,LEO (ISS),NASA (COTS),Success,No attempt
3,2012-10-08,0:35:00,F9 v1.0 B0006,CCAFS LC-40,SpaceX CRS-1,500,LEO (ISS),NASA (CRS),Success,No attempt
4,2013-03-01,15:10:00,F9 v1.0 B0007,CCAFS LC-40,SpaceX CRS-2,677,LEO (ISS),NASA (CRS),Success,No attempt


### 3. Total payload carried for NASA (CRS)

In [4]:
q("SELECT SUM(PAYLOAD_MASS__KG_) AS total_kg FROM SPACEXTBL WHERE Customer = 'NASA (CRS)'")

,total_kg
0,45596


### 4. Average payload mass of F9 v1.1

In [5]:
q("SELECT AVG(PAYLOAD_MASS__KG_) AS avg_kg FROM SPACEXTBL WHERE Booster_Version = 'F9 v1.1'")

,avg_kg
0,2928.4


### 5. First successful ground-pad landing

In [6]:
q("SELECT MIN(Date) AS first_ground_landing FROM SPACEXTBL WHERE Landing_Outcome = 'Success (ground pad)'")

,first_ground_landing
0,2015-12-22


### 6. Drone-ship successes with payload between 4000 and 6000 kg

In [7]:
q("SELECT Booster_Version FROM SPACEXTBL WHERE Landing_Outcome = 'Success (drone ship)' AND PAYLOAD_MASS__KG_ > 4000 AND PAYLOAD_MASS__KG_ < 6000")

,Booster_Version
0,F9 FT B1022
1,F9 FT B1026
2,F9 FT B1021.2
3,F9 FT B1031.2


### 7. Successful and failed mission outcomes

In [8]:
q("SELECT CASE WHEN Mission_Outcome LIKE 'Success%' THEN 'Success' ELSE Mission_Outcome END AS outcome, COUNT(*) AS total FROM SPACEXTBL GROUP BY 1")

,outcome,total
0,Failure (in flight),1
1,Success,100


### 8. Boosters that carried the maximum payload

In [9]:
q("SELECT DISTINCT Booster_Version FROM SPACEXTBL WHERE PAYLOAD_MASS__KG_ = (SELECT MAX(PAYLOAD_MASS__KG_) FROM SPACEXTBL)")

,Booster_Version
0,F9 B5 B1048.4
1,F9 B5 B1049.4
2,F9 B5 B1051.3
3,F9 B5 B1056.4
4,F9 B5 B1048.5
5,F9 B5 B1051.4
6,F9 B5 B1049.5
7,F9 B5 B1060.2
8,F9 B5 B1058.3
9,F9 B5 B1051.6


### 9. 2015 drone-ship landing failures

In [10]:
q("SELECT Date, Booster_Version, Launch_Site FROM SPACEXTBL WHERE Landing_Outcome = 'Failure (drone ship)' AND substr(Date,1,4) = '2015'")

,Date,Booster_Version,Launch_Site
0,2015-01-10,F9 v1.1 B1012,CCAFS LC-40
1,2015-04-14,F9 v1.1 B1015,CCAFS LC-40


### 10. Rank landing outcomes between 2010-06-04 and 2017-03-20

In [11]:
q("SELECT Landing_Outcome, COUNT(*) AS n FROM SPACEXTBL WHERE Date BETWEEN '2010-06-04' AND '2017-03-20' GROUP BY Landing_Outcome ORDER BY n DESC")

,Landing_Outcome,n
0,No attempt,10
1,Success (drone ship),5
2,Failure (drone ship),5
3,Success (ground pad),3
4,Controlled (ocean),3
5,Uncontrolled (ocean),2
6,Failure (parachute),2
7,Precluded (drone ship),1
